# Block 6 · Neural networks & responsible ML
*Which model should a bank use, and what do its decisions do to each group of people?*

> Data Mining · SGH Warsaw School of Economics (223121-0090)
>
> Version 2, updated 4 Nov 2026 for the class on 10 Nov. If Experiment 1 below is not called "Bend the boundary", you have an older copy: update the course folder (see the setup guide).

Block 5's large language models are neural networks. Tonight: what a neural network is, and what a model's decisions do to people. Two parts:
1. Networks: what a neuron and a network compute, how a network is trained and when it is stopped, and whether it beats the course's models on our data.
2. People: how to audit a model's decisions by group, and what the law requires.

**The methods you practise tonight**
1. **A neural network** - a **neuron** takes a weighted sum of its inputs plus a bias and puts it through an activation. A **weight** says how much an input counts; the **bias** is a number added to every score; the **activation** is the function applied to that sum, z. The **sigmoid**, 1 / (1 + e^-z), gives p between 0 and 1: one neuron with a sigmoid is Block 2's logistic regression. **ReLU** = max(0, z): keep z if it is positive, else 0. A **hidden layer** is a layer of neurons (**hidden units**) between the inputs and the output; it makes new columns from the inputs. A network with one or more hidden layers is a **multilayer perceptron (MLP)**. A **parameter** is a number that training chooses: every weight and every bias.
2. **Training** - the **loss** (Block 2's log-loss) is one number that says how wrong the p are; lower is better. The **slope** says how fast the loss changes as a weight grows; all the slopes of a network's weights together are the **gradient**, and walking downhill on the loss step by step (new w = w - learning rate × slope) is **gradient descent**. **Backpropagation** finds every weight's slope at once, from the output back; the software does it. The **learning rate** is the size of each step: too small is slow; too large makes the steps and the loss blow up (**exploding gradients**; the cure is a smaller learning rate) or leaves a **dead unit**, a ReLU unit that outputs 0 for every row and never learns again. Each step uses a **mini-batch**, a small set of rows (200 in scikit-learn); steps on random mini-batches are **stochastic gradient descent**; one pass over all training rows is an **epoch**. The **validation part** is training rows held out from fitting. **Early stopping** stops when the validation score stops improving and keeps the best epoch's weights; the **patience** is the number of epochs in a row without improvement after which training stops. A **leash** is a setting that limits how flexible a model can get (Block 2): early stopping is the network's everyday leash; two more are the **width** (the number of hidden units) and **weight decay** (`alpha`, a penalty on large weights).
3. **Choosing a model** - every model of the course on the same split, chosen by 5-fold cross-validation (CV) AUC on the training clients; the test set is read once, at the end. **GBDT** = gradient-boosted decision trees: Block 3's relay of small trees, each fixing what the trees before still get wrong. Besides AUC: **calibration** (are the p honest probabilities? of the clients given 20%, about 20% should default), **reasons** (can a declined client be told, in plain words, which facts drove the decision?) and **monotonicity** (a column that signals more risk never lowers the score when it grows).
4. **An audit by group** - a model that gives each client a probability of default (PD), the probability that the client will not pay, is a **PD model**. A **protected attribute** is a trait a decision must not treat people worse for: sex, age, ethnic origin, religion, disability. An audit by group counts what a rule does inside each group. Step 1: each group's **base rate** (its actual share of the event, here default) and **decline rate** (its share declined). Step 2, **compare the gaps**: the decline-rate gap against the base-rate gap, in percentage points (the difference between two percentages: 50% - 20% = 30 points); a decline gap minus base-rate gap above 0 means the rule enlarges the difference. Step 3, the errors: the **false-alarm rate** (FPR: good applicants declined / all good applicants) and the **miss rate** (FNR: defaulters approved / all defaulters). Step 4: **precision** (PPV: defaulters among the declined) and **calibration by group** (each group's mean p against its base rate, a first check). On a sample every gap has its **noise** (±): how far it could move by chance in another sample of the same size (a 95% range). The **impossibility** result: when base rates differ, honest scores and equal error rates cannot in general both hold. **Fairness through unawareness** = removing the protected column from the model; it does not remove the differences, because other columns carry part of it (**proxies**). A **feedback loop**: decisions shape the data the next model learns from.


**Our example data.** Block 2's table, loaded by Block 3's file `block3_tools.py`: 30,000 credit-card clients of a Taiwanese bank, observed for six months in 2005 (April to September). The target is `default` (1 = the client did not pay next month, 0 = repaid). The inputs are Block 2's 12 columns:

| name | meaning |
|---|---|
| `limit_bal` | credit limit: the most the client may owe on the card, in New Taiwan dollars |
| `age` | age in years |
| `female` | 1 = female, 0 = male |
| `edu_graduate` | 1 = graduate school |
| `edu_high_school` | 1 = high school |
| `edu_other` | 1 = other or unknown education (0 in all three education columns = university) |
| `married` | 1 = married |
| `pay_delay_recent` | September repayment status: -2 / -1 / 0 = nothing due / paid in full / paid at least the minimum; 1-8 = months late |
| `months_delayed_6m` | in how many of the last six months the client was late |
| `utilisation` | September bill / credit limit: the share of the limit used |
| `pay_ratio` | September payment / August bill |
| `bill_trend` | (September bill - April bill) / credit limit |

The split is Block 2's too: 21,000 **training** clients and 9,000 **test** clients (70/30, stratified, `random_state=42`), as in Blocks 2-4. Models are chosen on the training clients only; the test clients are read for models already fixed. The rule of the audit is Block 3's: decline when p > 0.25 (a missed default costs as much as 3 wrong declines). `female` and `age` are protected attributes: the audit reads them for every test client.

**The small worlds** come first, small enough to compute everything by hand: **clients 1-4** (one neuron and its loss), **the four points** (a network by hand), **one weight** (steps downhill), **ten epochs** (when to stop) and **twenty applicants** in groups A and B (an audit by hand) are all made up; **Block 3's crescents** (200 points in two classes, and 20,000 new ones) are generated by a formula. The data only illustrate the methods; the same steps work on any table with a 0/1 target.


**How we use this notebook.** The lecture stops five times for an **experiment**: the slide names it ("Experiment 1" ... "Experiment 5"); run that section. Each experiment has four parts:
* a **card** - the question, the one value you change, and what to look for (the same text as on the slide);
* a **plain cell** - change the value marked `# change me`, then re-run the cell (Shift+Enter);
* a **slider cell** - the same picture with a slider or a dropdown; it redraws when you let go of it. No slider? Use the plain cell;
* **The code behind it (for your project)** - the few scikit-learn lines behind the picture, explained, and a cell that runs them; read it at home.

**Tonight's experiments:** [1 - Bend the boundary](#Experiment-1---Bend-the-boundary) · [2 - Set the learning rate](#Experiment-2---Set-the-learning-rate) · [3 - Stop early](#Experiment-3---Stop-early) · [4 - Audit by group](#Experiment-4---Audit-by-group) · [5 - Remove a column](#Experiment-5---Remove-a-column). The table of contents (JupyterLab: the left sidebar; Colab: the list icon on the left) jumps to any section.

**Demos 1-5** show the slides' ideas computed on our data; replay them at home. The functions they call live in `block6_tools.py`, next to this notebook (they are the names with `lab.` in front); the slides are drawn by the same functions, so you see the same pictures. Some cells need a few seconds the first time - they print "Computing ... (about N s, only the first time)" - and are instant after that; N is for a recent laptop, Colab can take two or three times as long. The homework is at the end: A by hand, B in code.

* `[*]` next to a cell means it is still running. The prepare cell takes under a minute.
* A `NameError` (for example, name 'lab' is not defined) usually means the setup cell was not run, also after a restart: run the setup cell, then your cell again. To run everything above: JupyterLab *Run → Run All Above Selected Cell*; Colab *Runtime → Run before*.
* Decimals take a dot: 0.3. In Python, 0,3 means something else (a pair of numbers). In an answer cell, `cost=0,04` stops the cell with a `SyntaxError` about "leading zeros", and inside a list `[0,731]` is read as two numbers, 0 and 731: write 0.04 and 0.731.
* Text values take quotes: `"sex"`, not `sex`; without quotes Python stops with `NameError: name 'sex' is not defined`.
* **Google Colab** (nothing to install): open this notebook from GitHub, run the next cell (it fetches the three files and the course data below), then run the setup cell; nothing else changes. If the sliders do not appear, use the plain cells.

```
!wget -nv -O finance_data.py https://raw.githubusercontent.com/wlazlod/didactics/main/data-mining/lecture_1/finance_data.py
!wget -nv -O block3_tools.py https://raw.githubusercontent.com/wlazlod/didactics/main/data-mining/lecture_3/block3_tools.py
!wget -nv -O block6_tools.py https://raw.githubusercontent.com/wlazlod/didactics/main/data-mining/lecture_6/block6_tools.py
!mkdir -p ../data && wget -nv -O ../data/taiwan_credit.csv https://raw.githubusercontent.com/wlazlod/didactics/main/data-mining/data/taiwan_credit.csv
```

Run the setup cell first: it must print `Setup OK`, then `Sliders OK` (or a line saying that you will use the plain cells - that is fine too). A line starting with "Note:" about the scikit-learn version needs no action; only a WARNING needs the fix it names. Then run the prepare cell below it.

In [ ]:
# Google Colab only: fetch the course files and the course data (on your laptop this cell does nothing).
# Run it once, then run the setup cell below.
import sys
if "google.colab" in sys.modules:
    !wget -nv -O finance_data.py https://raw.githubusercontent.com/wlazlod/didactics/main/data-mining/lecture_1/finance_data.py
    !wget -nv -O block3_tools.py https://raw.githubusercontent.com/wlazlod/didactics/main/data-mining/lecture_3/block3_tools.py
    !wget -nv -O block6_tools.py https://raw.githubusercontent.com/wlazlod/didactics/main/data-mining/lecture_6/block6_tools.py
    !mkdir -p ../data && wget -nv -O ../data/taiwan_credit.csv https://raw.githubusercontent.com/wlazlod/didactics/main/data-mining/data/taiwan_credit.csv
else:
    print("Not on Colab: nothing to fetch. Run the setup cell below.")

In [ ]:
import sys
sys.path.append("../lecture_1")   # the course data loader, finance_data.py
sys.path.append("../lecture_3")   # Block 3's tools: the data, the split, the models, the crescents
sys.path.append(".")              # block6_tools.py, next to this notebook

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", None)   # long table cells in full
pd.set_option("display.width", 120)

# You do not need to read the next lines: they stop with a clear message
# if a file is missing.
try:
    import block6_tools as lab    # tonight's experiments: one function each
except ModuleNotFoundError as error:   # a file is missing: say which one, and stop
    if error.name not in ("block6_tools", "block3_tools", "finance_data"):
        raise
    import warnings; warnings.filterwarnings("ignore", "To exit")
    raise SystemExit(f"{error.name}.py was not found. Colab: run the cell "
                     "above first (if you did, check that wget printed no "
                     "ERROR line). Laptop: update the course folder (git pull, "
                     "see the setup guide).") from None

# The data and the split of Block 2: 12 columns; 70% training, 30% test,
# stratified, random_state=42. Models are chosen on the training clients; the
# test clients are read for models already fixed. These are copies: changing
# them in your own cells does not change what the lab keeps.
X_train, X_test, y_train, y_test = lab.course_split()   # 21,000 / 9,000 clients; default 1 / 0

# Prints "Setup OK: ...", a Note about the scikit-learn version if there is one
# (no action needed; only a WARNING needs the fix it names), then "Sliders OK"
# or a line saying that you will use the plain cells.
lab.setup_check()

## Prepare tonight's experiments
Run it once now: it computes the training runs and the audit models that Experiments 2-5 draw - under a minute (about 20 s on a recent laptop, longer on Colab). It prints one line per part with the seconds it took, then "Ready". After it, every experiment redraws at once. If you skip it, each experiment computes its own part the first time you run it.

In [ ]:
lab.prepare()   # Experiments 2-5: the learning rates, the 100 epochs, our GBDT and its four refits

## 1 One neuron, four clients (slides 7-10)
Four made-up clients numbered 1-4, two numbers each: x1 = months late in the last 6 months, x2 = the share of the credit limit used. One neuron with the weights of the slides, chosen by hand: w1 = 1, w2 = 2, bias b = -3. Step 1, the weighted sum plus the bias: z = w1·x1 + w2·x2 + b, one score per client. Step 2, the activation: the sigmoid turns z into p = 1 / (1 + e^-z), a number between 0 and 1. The first table shows both steps for each client; the second, the sigmoid for z from -3 to 3 (the homework uses it). Try other weights in the first line. `lab.CLIENTS` is a copy: changing it changes nothing in the lab.

In [ ]:
display(lab.neuron_table(w1=1, w2=2, b=-3))   # change me: other weights; steps 1-2 for clients 1-4
lab.sigmoid_table()                            # the sigmoid: p for z = -3 ... 3

## Demo 1 - A network with no hidden layer is logistic regression (replay at home: slide 10)
One neuron with a sigmoid is Block 2's logistic regression: the bias is the intercept b0, the weights are the coefficients. In scikit-learn it is a network with no hidden layer, `MLPClassifier(hidden_layer_sizes=())`, in a pipeline after a scaler. The demo scores it and logistic regression by 5-fold cross-validation on our 21,000 training clients (Block 3's five folds) and prints both CV AUCs (mean and sd, the standard deviation of the five folds' AUCs: how much they vary from fold to fold). Then it compares their **out-of-fold** p - each client's p from the model fitted on the other four folds, a model that did not learn from that client (Block 3): how closely the two agree (their correlation, 1 = in perfect step) and the largest difference for one client. About 2 s the first time.

In [ ]:
lab.no_hidden_layer()   # both CV AUCs, then how closely their out-of-fold p agree

## 2 A network by hand: four points (slides 11-16)
Four made-up points: x1 and x2 are each 0 or 1, and the class is 1 when exactly one of them is 1. No straight line separates them, so one neuron fails. The network of the slides has a hidden layer of two ReLU units (ReLU = max(0, z): keep z if it is positive, else 0). Step 1, the hidden units: h1 = ReLU(x1 - x2), h2 = ReLU(x2 - x1). Step 2, the output neuron: z = 4·h1 + 4·h2 - 2, then the sigmoid; the class is 1 if p > 0.5. The first table shows both steps; the second, the same network without ReLU (each hidden unit passes its sum on unchanged). Step 3 counts the parameters - every weight and every bias - of the four points' network, of our network (12 inputs, one hidden layer of 16 units) and of the book's network for handwritten digits (784 pixels, hidden layers of 256 and 128 units, 10 outputs).

In [ ]:
display(lab.xor_table())             # steps 1-2 with ReLU: is each class right?
display(lab.xor_table(relu=False))   # without ReLU
lab.count_parameters(2, (2,))        # step 3: the four points' network
lab.count_parameters(12, (16,))      # our network: 12 inputs, 16 hidden units, one output
lab.count_parameters(784, (256, 128), 10);   # the book's digits network (the ; hides the repeated total)

## Experiment 1 - Bend the boundary
*About 7 minutes.* Block 3's crescents: 200 training points in two classes. A network with one hidden layer is fitted to them; the shading shows where it predicts the red class (p > 0.5).

**Question:** From how many hidden units does the boundary bend around the crescents, and what happens with very many units?

**Change:** `hidden_units`: 0 (one neuron, no hidden layer) to 64.

**Look for:** the boundary between the shades, and the two accuracies in the title: on the 200 training points and on 20,000 new points.

In [ ]:
hidden_units = 1   # change me: 0 to 64 (0, 1, 2, 3, 4, 8, 16, 32 or 64)
lab.bend_boundary(hidden_units)

In [ ]:
lab.slider(lab.bend_boundary, hidden_units=lab.E1_UNITS)   # a slider over 0, 1, 2, 3, 4, 8, 16, 32, 64

### The code behind it (for your project)
```python
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
X, y, X_new, y_new = lab.crescents()
net = make_pipeline(StandardScaler(),
                    MLPClassifier(hidden_layer_sizes=(16,), solver="lbfgs",
                                  max_iter=5000, random_state=42))
net.fit(X, y)
net.score(X, y), net.score(X_new, y_new)
net[-1].coefs_, net[-1].intercepts_
```
* `lab.crescents()`: Block 3's crescents: `X` = 200 points × 2 columns, `y` = their classes (0 or 1); `X_new`, `y_new` = 20,000 new points from the same formula.
* `make_pipeline(StandardScaler(), MLPClassifier(...))`: the scaler and the network in one object, as Block 2's logistic regression. The steps of training need columns on one scale.
* `hidden_layer_sizes=(16,)`: one hidden layer of 16 units; `(16, 8)` would be two hidden layers, `()` none (one neuron). Inside, the units use ReLU (`activation="relu"`, the default); the output uses the sigmoid.
* `solver="lbfgs"`: the method that finds the weights. With 200 rows it can use all of them in every step and finds a low loss quickly; for our 21,000 clients the default, `"adam"`, takes small steps on mini-batches (section 3). `max_iter=5000` allows enough steps; `random_state=42` fixes the random starting weights, so the same network comes back every time.
* `score`: the accuracy, the share right (the crescents' two classes are equally common, so accuracy is fine here).
* `net[-1]`: the pipeline's last step, the network. `coefs_`: one table of weights per layer (inputs × units); `intercepts_`: the biases, one per unit.

The cell fits the network with 16 hidden units, prints both accuracies, the shapes of the weight and bias tables, and counts the parameters.

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
X, y, X_new, y_new = lab.crescents()     # 200 training points, 20,000 new points
net = make_pipeline(StandardScaler(),
                    MLPClassifier(hidden_layer_sizes=(16,), solver="lbfgs",
                                  max_iter=5000, random_state=42))
net.fit(X, y)
print(f"accuracy: training points {net.score(X, y):.3f}, new points {net.score(X_new, y_new):.3f}")
mlp = net[-1]                            # the network, the pipeline's last step
print("weight tables (inputs x units):", [w.shape for w in mlp.coefs_])
print("biases (one per unit):", [len(b) for b in mlp.intercepts_])
print("parameters:", sum(w.size for w in mlp.coefs_) + sum(b.size for b in mlp.intercepts_))

## 3 Training by hand (slides 20-24)
Training starts from small random weights and repeats small steps. Step 1, the **loss** of clients 1-4 with made-up outcomes (1 and 2 repaid, 3 and 4 defaulted): Block 2's log-loss, -ln p for a defaulter and -ln(1 - p) for a client who repaid; the mean is one number for all four. Steps 2-3 on one made-up weight w with the loss L(w) = (w - 3)², lowest at w = 3: its slope is 2(w - 3), and each step moves w against the slope, new w = w - learning rate × slope, starting from w = 0. The cell prints three steps and the loss after each; try the learning rates 0.05, 0.25 and 1.1. Step 4: each step of a real network uses one mini-batch of rows, so one epoch, one pass over all rows, takes rows / batch size steps.

In [ ]:
display(lab.loss_table())         # step 1: each client's loss and the mean
learning_rate = 0.25              # change me: 0.05, 0.25 or 1.1
lab.gradient_steps(lr=learning_rate)   # steps 2-3: three steps downhill from w = 0
steps = lab.count_steps(21_000, 200)   # step 4: steps in one epoch, batches of 200

## Experiment 2 - Set the learning rate
*About 7 minutes.* Our 21,000 training clients and tonight's network (16 hidden units), trained for 50 epochs with mini-batch steps (slide 24) from the same starting weights; only the learning rate changes.

**Question:** Which learning rates barely move the loss, which bring it down fast, and which make the training fail?

**Change:** `learning_rate`: 0.00001, 0.001, 0.03, 0.3, 3, 10 or 20.

**Look for:** the training loss after each epoch (lower is better; the rates you tried stay in grey) and, in the title, how many of the 16 hidden units are dead.

In [ ]:
learning_rate = 0.001   # change me: 0.00001, 0.001, 0.03, 0.3, 3, 10 or 20
lab.learning_rate(learning_rate)   # lab.learning_rate_reset() clears the rates you tried

In [ ]:
lab.slider(lab.learning_rate, learning_rate=lab.E2_RATES)   # a slider over the seven rates

### The code behind it (for your project)
```python
import warnings
from sklearn.exceptions import ConvergenceWarning
net = make_pipeline(StandardScaler(),
                    MLPClassifier(hidden_layer_sizes=(16,), solver="sgd", momentum=0,
                                  learning_rate_init=0.3, max_iter=50, random_state=42))
with warnings.catch_warnings():
    warnings.simplefilter("ignore", ConvergenceWarning)
    net.fit(X_train, y_train)
net[-1].loss_curve_
```
* `solver="sgd"`: stochastic gradient descent, the plain steps of slides 22-24: each step uses one random mini-batch of 200 rows (`batch_size`, by default 200, or all rows if there are fewer). `momentum=0` switches off a speed-up that scikit-learn adds by default, so every step is exactly new w = w - learning rate × slope. The default solver, `"adam"`, also adapts the step size as it goes.
* `learning_rate_init=0.3`: the learning rate (the default is 0.001).
* `max_iter=50`: at most 50 epochs (for `"sgd"` and `"adam"`, `max_iter` counts epochs).
* The warning: scikit-learn stops on its own when the loss improves by less than `tol` = 0.0001 for more than 10 epochs in a row; when it reaches `max_iter` first, it warns that the training has not converged (not finished). We stop at 50 epochs on purpose, so the `with` block hides that one warning, and only inside it.
* `loss_curve_`: the training loss after each epoch - the curve of Experiment 2.

The cell trains at learning rate 0.3 and draws the curve.

In [ ]:
import warnings
from sklearn.exceptions import ConvergenceWarning
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
net = make_pipeline(StandardScaler(),
                    MLPClassifier(hidden_layer_sizes=(16,), solver="sgd", momentum=0,
                                  learning_rate_init=0.3, max_iter=50, random_state=42))
with warnings.catch_warnings():
    # scikit-learn would warn that 50 epochs did not meet its own stopping rule - fine here
    warnings.simplefilter("ignore", ConvergenceWarning)
    net.fit(X_train, y_train)
curve = net[-1].loss_curve_          # the training loss after each epoch
print(f"{len(curve)} epochs: training loss {curve[0]:.3f} after epoch 1, {curve[-1]:.3f} after epoch 50")
fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(range(1, len(curve) + 1), curve)
ax.set_xlabel("epoch")
ax.set_ylabel("training loss")
ax.set_title("learning rate 0.3: the training loss after each epoch")
plt.show()

## 4 When to stop (slides 27-28)
Step 5 of training. Ten made-up epochs: after each, the training loss and the AUC on a **validation part** (training rows held out from fitting, as in Block 3). `lab.EPOCH_TOY` holds them. **Early stopping** watches the validation score after each epoch and counts the epochs in a row without improvement; when the count reaches the **patience**, training stops and the best epoch's weights are kept. The cell walks through the ten epochs with the patience you choose (scikit-learn stops at the 11th epoch in a row without a gain, `n_iter_no_change=10`).

In [ ]:
display(lab.EPOCH_TOY)           # ten made-up epochs
patience = 3                     # change me: 1 to 10
lab.patience_walk(patience=patience)

## Experiment 3 - Stop early
*About 8 minutes.* Our training clients: 16,800 to fit and 4,200 held out for validation. A wide network (two hidden layers of 256 units, 69,377 parameters) with no other leash, trained for 100 epochs.

**Question:** After which epoch would you stop, and how much validation AUC does training on to epoch 100 cost?

**Change:** `stop_after`: an epoch from 1 to 100.

**Look for:** left, the training loss (lower is better); right, the validation AUC (higher is better), both drawn up to your epoch; the dot marks the best epoch so far, and the title gives its values.

In [ ]:
stop_after = 5   # change me: an epoch from 1 to 100
lab.stop_early(stop_after)

In [ ]:
lab.slider(lab.stop_early, stop_after=lab.E3_EPOCHS)   # a slider over the epochs 1 to 100

### The code behind it (for your project)
```python
net = make_pipeline(StandardScaler(),
                    MLPClassifier(hidden_layer_sizes=(16,), early_stopping=True,
                                  max_iter=500, random_state=42))
net.fit(X_train, y_train)
mlp = net[-1]
mlp.n_iter_, mlp.validation_scores_, mlp.best_validation_score_
```
* `early_stopping=True`: scikit-learn holds out 10% of the training rows (`validation_fraction=0.1`, the same share of defaulters as in the rest), scores them after each epoch, stops when more than 10 epochs in a row (`n_iter_no_change=10`) bring no gain of at least `tol` = 0.0001 (the deck's network: best at epoch 24, stopped after epoch 35, the 11th without a gain), and puts the best epoch's weights back.
* `max_iter=500`: only a ceiling on the epochs; early stopping decides.
* `n_iter_`: the epochs actually run; `validation_scores_`: the validation score after each epoch; `best_validation_score_`: the best of them.
* scikit-learn's validation score is the **accuracy** at p = 0.5 (the share right when everyone with p above 0.5 is called a defaulter), not AUC; it has no AUC option here. For a 22% event accuracy is a coarse score, so Experiment 3 draws the AUC with its own loop.
* The loop: `partial_fit` trains exactly one more epoch per call (`classes=[0, 1]` names the two classes the first time); after each epoch, the AUC on the validation part. Experiment 3's wide network: `hidden_layer_sizes=(256, 256)`, two hidden layers of 256 units, and `alpha=0.0`, no weight decay (another leash), so nothing but the number of epochs holds it back. The scaler is fitted on the 16,800 rows that are fitted, never on the validation part.

The cell does both: the deck's network with early stopping, then the first 10 epochs of Experiment 3's wide network (about 1 s).

In [ ]:
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
net = make_pipeline(StandardScaler(),
                    MLPClassifier(hidden_layer_sizes=(16,), early_stopping=True,
                                  max_iter=500, random_state=42))
net.fit(X_train, y_train)
mlp = net[-1]
best = int(np.argmax(mlp.validation_scores_)) + 1      # Python counts from 0, epochs from 1
print(f"ran {mlp.n_iter_} epochs; best validation accuracy {mlp.best_validation_score_:.4f} "
      f"after epoch {best}, whose weights are kept\n")

# Experiment 3's wide network: one epoch per partial_fit, then the AUC on the validation part
X_fit, X_val, y_fit, y_val = train_test_split(X_train, y_train, test_size=0.2,
                                              stratify=y_train, random_state=42)
scaler = StandardScaler().fit(X_fit)                   # fitted on the 16,800 only
wide = MLPClassifier(hidden_layer_sizes=(256, 256), alpha=0.0, random_state=42)
for epoch in range(1, 11):
    wide.partial_fit(scaler.transform(X_fit), y_fit, classes=[0, 1])
    auc = roc_auc_score(y_val, wide.predict_proba(scaler.transform(X_val))[:, 1])
    print(f"epoch {epoch:2d}: training loss {wide.loss_:.3f}, validation AUC {auc:.3f}")

## Demo 2 - Our network on our data (replay at home: slides 31-32)
Tonight's network, built as Block 2's logistic regression was: a scaler and the model in one pipeline, `make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(16,), early_stopping=True, max_iter=500, random_state=42))` (`lab.network()` gives it, unfitted). The demo prints its number of parameters, after which epoch early stopping ended and which epoch's weights it kept, and its 5-fold CV AUC on the 21,000 training clients. About 3 s the first time.

In [ ]:
lab.our_network()

## Demo 3 - The comparison (replay at home: slides 33-36, Backup 10-12)
Every model of the course on the same split, judged by the same metric: logistic regression (Block 2), a decision tree of depth 5, a random forest of 300 trees, **GBDT** (gradient-boosted decision trees: Block 3's relay of small trees, each fixing what the trees before still get wrong; `HistGradientBoostingClassifier`) and tonight's network (MLP). For each: the 5-fold CV AUC on the 21,000 training clients (mean and sd), the number that chooses; the **calibration gap** of its out-of-fold p (the 21,000 in ten equal-size groups by p; in each group the mean p against the share who defaulted; the ten gaps averaged, in percentage points: small = honest p); then the test AUC, read once, for models already chosen. The figure is slide 34's. Its "Computing" line prints the measured time (about 10 s on a laptop).

In [ ]:
lab.tournament()   # the figure, then the table

## 5 Twenty applicants (slides 38-45)
An audit small enough to count by hand. Twenty made-up applicants in groups A and B of ten, each with a model's score; the scores come from a model that does not use the group. Group A: eight applicants with 0.25 (two default) and two with 0.50 (one defaults); group B: five with 0.40 (two default) and five with 0.80 (four default). The rule (made up): decline when the score is above 0.45. The cell prints the four steps of the audit, in the order of the slides: 1 decline rates and base rates; 2 the two gaps and their difference; 3 each group's 2 × 2 table (rows: what happened, columns: what the rule did) with its false-alarm rate and miss rate; 4 precision, the mean score and the base rate. `lab.APPLICANTS` holds the twenty rows.

In [ ]:
lab.toy_audit()   # steps 1-4 of the audit by group

## Experiment 4 - Audit by group
*About 8 minutes.* Our 9,000 test clients, our GBDT, and Block 3's rule: decline when p > 0.25. The audit compares two groups of clients.

**Question:** For which split is the decline-rate gap clearly larger than the base-rate gap, and whose good clients are wrongly declined more often?

**Change:** `group`: "sex", "age: under 25" or "age: 50 and over".

**Look for:** left, each group's base rate (grey) and decline rate (red), with both gaps, and the decline gap minus the base-rate gap with its noise (±), in the title; right, each group's two error rates: good clients wrongly declined and defaulters missed.

In [ ]:
group = "sex"   # change me: "sex", "age: under 25" or "age: 50 and over"
lab.audit_by_group(group)   # the figure, then the table and its three lines

In [ ]:
lab.slider(lab.audit_by_group, group=lab.E4_GROUPS)   # a dropdown of the three splits

### The code behind it (for your project)
```python
gbdt = lab.model("boosting").fit(X_train, y_train)
p = gbdt.predict_proba(X_test)[:, 1]
lab.group_report(y_test, p, X_test["female"] == 1, names=("women", "men"), threshold=0.25)
```
* `lab.model("boosting")`: Block 3's GBDT (`HistGradientBoostingClassifier` with Block 3's settings), fitted on the 21,000 training clients with all 12 columns. `p` = its probability of default for each of the 9,000 test clients: a model already fixed, so nothing is chosen here.
* `X_test["female"] == 1`: True for women, False for men, one value per test client; `names` gives the two groups' names in that order. By age: `X_test["age"] < 25` with `names=("under 25", "25 and over")`.
* `threshold=0.25`: decline when p > 0.25 (Block 3's rule).
* The table has one row per group: n, base rate, mean p, decline rate, false-alarm rate, miss rate and precision, in %. Under it: the decline-rate gap and the base-rate gap, each with its noise, 1.96 × sqrt(r1(1 - r1)/n1 + r2(1 - r2)/n2) for two rates r1, r2 on n1 and n2 clients; then the decline gap minus the base-rate gap with its own noise: for each client d = 1 if declined, minus 1 if defaulted (so +1, 0 or -1), and the noise is 1.96 × sqrt(var(d1)/n1 + var(d2)/n2).

The cell runs `group_report`, then computes the same numbers with a few lines of pandas. **Use it on your project (homework B1).**

In [ ]:
gbdt = lab.model("boosting").fit(X_train, y_train)    # Block 3's GBDT, all 12 columns
p = gbdt.predict_proba(X_test)[:, 1]                  # p for each of the 9,000 test clients
display(lab.group_report(y_test, p, X_test["female"] == 1, names=("women", "men"), threshold=0.25))

# The same numbers in pandas
declined = pd.Series(p > 0.25, index=X_test.index)    # True = declined
defaulted = y_test == 1                               # True = defaulted
women, men = X_test["female"] == 1, X_test["female"] == 0
rows = {}
for name, in_group in [("women", women), ("men", men)]:
    dec, dft = declined[in_group], defaulted[in_group]
    rows[name] = {"n": in_group.sum(),
                  "base rate (%)": 100 * dft.mean(),             # share who defaulted
                  "mean p (%)": 100 * p[in_group.to_numpy()].mean(),
                  "decline rate (%)": 100 * dec.mean(),          # share declined
                  "false-alarm rate (%)": 100 * dec[~dft].mean(),   # good clients declined / all good clients
                  "miss rate (%)": 100 * (~dec)[dft].mean(),     # defaulters approved / all defaulters
                  "precision (%)": 100 * dft[dec].mean()}        # defaulters among the declined
display(pd.DataFrame(rows).T.round(1).astype({"n": int}))
gap = (declined[women].mean() - declined[men].mean()) - (defaulted[women].mean() - defaulted[men].mean())
d = declined.astype(int) - defaulted.astype(int)      # +1, 0 or -1 per client
noise = 1.96 * np.sqrt(d[women].var() / women.sum() + d[men].var() / men.sum())
print(f"decline gap minus base-rate gap: {100 * gap:+.1f} points (± {100 * noise:.1f})")

## Demo 4 - Can the other columns guess sex? (replay at home: slide 49, Backup 16)
Removing `female` from the model helps only if the other columns cannot stand in for it. A column that carries part of another is a **proxy**. The demo trains a GBDT to guess `female` from the other 11 columns and scores it by 5-fold CV AUC on the training clients (0.5 = no better than a coin, 1 = always right). Then it prints the means of five columns for women and for men. About 5 s the first time.

In [ ]:
lab.proxy_check()

## Experiment 5 - Remove a column
*About 7 minutes.* Our GBDT trained again without some columns, then the same 9,000 test clients and the same rule: decline when p > 0.25.

**Question:** Once female, or age, is removed from the model, do the groups get equal decline rates?

**Change:** `removed`: "nothing", "female", "female, married" (married differs by sex and age), "female, age" or "female, married, age".

**Look for:** left by sex, right by age: each group's base rate (grey) and decline rate (red); in the title, the test AUC and how many of the 9,000 decisions changed.

In [ ]:
removed = "nothing"   # change me: "nothing", "female", "female, married", "female, age" or "female, married, age"
lab.remove_column(removed)

In [ ]:
lab.slider(lab.remove_column, removed=lab.E5_REMOVED)   # a dropdown of the five choices

### The code behind it (for your project)
```python
small = lab.model("boosting").fit(X_train.drop(columns=["female"]), y_train)
p_new = small.predict_proba(X_test.drop(columns=["female"]))[:, 1]
((p > 0.25) != (p_new > 0.25)).sum()
```
* `X_train.drop(columns=["female"])`: the training clients without the column. The test clients must lose the same column, because a model predicts from exactly the columns it was trained on.
* `X_test["female"]` itself stays: the audit reads each client's group from the test data, whether or not the model uses that column.
* `(p > 0.25) != (p_new > 0.25)`: True where one model declines and the other approves; `.sum()` counts the decisions that changed.

The cell fits the GBDT with and without `female`, prints both test AUCs and the decisions that changed, and audits the model without `female` by sex.

In [ ]:
from sklearn.metrics import roc_auc_score
gbdt = lab.model("boosting").fit(X_train, y_train)                             # all 12 columns
p = gbdt.predict_proba(X_test)[:, 1]
small = lab.model("boosting").fit(X_train.drop(columns=["female"]), y_train)   # without female
p_new = small.predict_proba(X_test.drop(columns=["female"]))[:, 1]
changed = (p > 0.25) != (p_new > 0.25)       # True: the decision changed
print(f"test AUC {roc_auc_score(y_test, p):.4f} with female, {roc_auc_score(y_test, p_new):.4f} without; "
      f"{changed.sum()} of {len(p):,} decisions changed")
lab.group_report(y_test, p_new, X_test["female"] == 1, names=("women", "men"))   # the audit still reads female

## Demo 5 - The same audit for the network (replay at home: Backup 14)
Experiment 4's audit for tonight's network instead of the GBDT: the same 9,000 test clients and the same rule, decline when p > 0.25. The demo prints the network's test AUC and the share declined, then `group_report` by sex and by age (under 25 against 25 and over).

In [ ]:
lab.mlp_audit()

## Homework
Two kinds of items: **A** = practice by hand (a calculator is fine, and Python works as one) and **B** = code. Each A item repeats one of tonight's slides. Under each A item there is an answer cell: replace each `...` with your answer and run the cell; `lab.check` then says "right" or "not yet", with a hint (it never stops the notebook). `lab.check("A1")` with no answers lists what an item takes. The solutions are released later. Also answer the open exam-practice question from the last slides (3-5 sentences); a model answer will be in the solutions.

**The open exam-practice question.** A car insurer uses a model to flag new applicants for a manual check. On 2,000 test applicants it reports, by age:

| group | applicants | base rate (share who later made a claim) | flagged | false-alarm rate (share of applicants with no claim who were flagged) |
|---|---|---|---|---|
| under 30 | 500 | 20% | 30% | 15% |
| 30 and over | 1,500 | 10% | 12% | 6% |

(a) Compute the gap in flag rates and the gap in base rates. Does the model reproduce the base-rate difference or enlarge it?
(b) Name two more numbers you would compare between the groups, and say why.
(c) The insurer proposes to remove age from the model. Will the flag rates then be equal? Give one reason.

### A - by hand

**A1. One neuron** (slides 7-10). A sports club predicts which members leave next year. x1 = visits per week, x2 = years as a member; the neuron's weights are w1 = -1, w2 = -0.5 and its bias b = 1. Three members (`lab.HW_MEMBERS`): X (x1 = 0, x2 = 0), Y (1, 2) and Z (3, 2). (a) z for each member. (b) p for each (the sigmoid table of section 1, `lab.sigmoid_table()`, has the z you need). (c) The coefficients b0, b1, b2 of a logistic regression that gives the same p.

In [ ]:
# A1 - replace each ... with your answer, then run the cell. A list: [..., ..., ...]
lab.check("A1",
          z=...,              # z of X, Y, Z, as a list
          p=...,              # p of X, Y, Z (three decimals are enough)
          coefficients=...)   # b0, b1, b2 of the logistic regression with the same p

**A2. A network by hand** (slides 12-16). The four points of section 2, (0,0), (1,0), (0,1) and (1,1), and another network: h1 = ReLU(x1 + x2 - 1), h2 = ReLU(x1 + x2), z = -8·h1 + 4·h2 - 2, then the sigmoid; the class is 1 if p > 0.5. (a) h1, h2, z, p and the class at each point. (b) The number of parameters of this network. (c) The number of parameters of a network with 5 inputs, one hidden layer of 8 units and one output, and of one with 5 inputs, two hidden layers of 10 and 4 units and one output.

In [ ]:
# A2 - lists in the order (0,0), (1,0), (0,1), (1,1)
lab.check("A2",
          h1=..., h2=..., z=..., p=...,
          classes=...,           # the class at each point (1 if p > 0.5)
          parameters=...,        # this network
          parameters_8=...,      # 5 inputs, one hidden layer of 8 units
          parameters_10_4=...)   # 5 inputs, hidden layers of 10 and 4 units

**A3. Steps downhill** (slides 21-24). One weight w with the loss L(w) = (w - 2)², lowest at w = 2; its slope is 2(w - 2). Start at w = 0; each step: new w = w - learning rate × slope. (a) Three steps with each of the learning rates 0.1, 0.5 and 1.5, with the loss after each step. (b) Which rate is slow, which reaches the lowest point, which overshoots? (c) 6,000 rows in batches of 200: how many steps make one epoch?

In [ ]:
# A3 - w and the loss after steps 1, 2, 3, as lists
lab.check("A3",
          w_01=..., loss_01=...,   # learning rate 0.1
          w_05=..., loss_05=...,   # learning rate 0.5
          w_15=..., loss_15=...,   # learning rate 1.5
          slow=..., lowest=..., overshoots=...,   # each a learning rate
          steps=...)               # steps in one epoch

**A4. When to stop** (slides 27-28). Eight made-up epochs (`lab.HW_EPOCHS`):

| epoch | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 |
|---|---|---|---|---|---|---|---|---|
| training loss | 0.55 | 0.48 | 0.44 | 0.41 | 0.39 | 0.37 | 0.35 | 0.33 |
| validation AUC | 0.70 | 0.74 | 0.76 | 0.75 | 0.75 | 0.74 | 0.73 | 0.72 |

(a) The best epoch. (b) With a patience of 3 (stop once 3 epochs in a row have not improved), after which epoch does training stop, and whose weights are kept? (c) What does training on to epoch 8 cost?

In [ ]:
# A4 - epochs as whole numbers; the cost as a difference of validation AUCs
lab.check("A4",
          best_epoch=...,
          stops_after=...,   # the epoch after which training stops
          kept_epoch=...,    # whose weights are kept
          cost=...)          # the validation AUC lost by training on to epoch 8

**A5. An audit by hand** (slides 39-44). A bike-rental scheme asks some new members for a deposit (the decision) and records who returned a bike late (the event). Here the base rate is a group's share of members who were late, and the deposit rate its share asked for a deposit; the false-alarm rate = members asked for a deposit who were not late / all who were not late; the miss rate = members late but not asked / all who were late; precision = late members among those asked. (`lab.HW_SCHEME`):

| group | members | late | asked for a deposit | late among those asked |
|---|---|---|---|---|
| under 30 | 20 | 6 | 8 | 5 |
| 30 and over | 30 | 6 | 6 | 4 |

(a) The base rates and the deposit rates. (b) Compare the gaps: does the rule reproduce the base-rate difference or enlarge it? (c) The false-alarm rates and the miss rates. (d) The precision in each group.

In [ ]:
# A5 - percents (21.4 for 21.4%) or decimals (0.214); lists in the order under 30, 30 and over;
# gaps in percentage points, under 30 minus 30 and over
lab.check("A5",
          base_rate=..., deposit_rate=...,
          deposit_gap=..., base_gap=...,
          difference=...,      # the deposit gap minus the base-rate gap
          false_alarm=..., miss=...,
          precision=...)

**A6. Remove the column?** (slides 45, 49-51; open, 3-4 sentences). The scheme of A5 removes age from its model. Will the deposit rates by age become equal, and what would you check instead?

In [ ]:
# A6 - write your answer between the quotes (3-4 sentences); once it is there, the check prints the model answer's three points
answer_A6 = """
"""
lab.check("A6", answer=answer_A6)

### B - code
Every item works on our data. The next cell imports the tools and fits, once, the two models the items audit, each on the 21,000 training clients: `gbdt` (Block 3's GBDT, `lab.model("boosting")`) and `net` (tonight's network, `lab.network()`: the scaler and the MLP with 16 hidden units and early stopping); `p_gbdt` and `p_net` are their p for the 9,000 test clients. Also at hand: `lab.group_report(y_true, p, group, names, threshold)` (Experiment 4's code behind it) and `lab.folds()`, Block 3's five CV folds.

In [ ]:
# Homework B: the tools (each one is explained in a "code behind it" cell above)
from sklearn.model_selection import cross_val_score
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

gbdt = lab.model("boosting").fit(X_train, y_train)   # Block 3's GBDT, all 12 columns
net = lab.network().fit(X_train, y_train)            # tonight's network
p_gbdt = gbdt.predict_proba(X_test)[:, 1]            # p for the 9,000 test clients
p_net = net.predict_proba(X_test)[:, 1]
print(f"{len(p_gbdt):,} test clients; mean p {p_gbdt.mean():.3f} (GBDT), {p_net.mean():.3f} (network)")

**B1. Audit your project's model** (optional for the project). Practise on our data first: audit the GBDT by age, 50 and over against everyone younger, with `lab.group_report`, and read the table and its three lines. Then, if your data have a protected attribute (or another meaningful split), run `lab.group_report(y_test, p_test, mask, names, threshold)` on your model's test p: `mask` is True for one group and False for the other, `names` their names, `threshold` your decision rule. Write one paragraph for your limitations: the two gaps and their difference with its noise, the error rates, and whether the base-rate gap itself is legitimate. No fixed answer for your project.

In [ ]:
# B1 - your code
older = X_test["age"] >= 50                     # True: 50 and over
# lab.group_report(y_test, p_gbdt, older, names=("50 and over", "under 50"), threshold=0.25)
# then on your project: lab.group_report(your_y_test, your_p_test, your_mask, names=(..., ...), threshold=...)

**B2 (optional). A network in your comparison.** Add `make_pipeline(StandardScaler(), MLPClassifier((16,), early_stopping=True, random_state=42))` to your project's CV comparison: does it beat your tree ensemble? No fixed answer for your project. On our data first: the 5-fold CV AUC of the network and of the GBDT on the training clients, with `cross_val_score(..., cv=lab.folds(), scoring="roc_auc")`; is the difference larger than the spread between the folds?

In [ ]:
# B2 - your code
candidates = {"GBDT": lab.model("boosting"),
              "network": make_pipeline(StandardScaler(),
                                       MLPClassifier((16,), early_stopping=True, random_state=42))}
for name, candidate in candidates.items():
    pass   # scores = cross_val_score(candidate, X_train, y_train, cv=lab.folds(), scoring="roc_auc")
           # print the name, scores.mean() and scores.std()

**B3. Audit tonight's network.** Run `lab.group_report` on the network's test p (`p_net`) by sex and by age (under 25 against 25 and over), with the rule p > 0.25. Is the pattern the same as Experiment 4's for the GBDT?

In [ ]:
# B3 - your code
lab.group_report(y_test, p_net, X_test["female"] == 1, names=("women", "men"), threshold=0.25)
# by age: X_test["age"] < 25, names=("under 25", "25 and over")

## Keep learning - Block 6 reading
* **James, Witten, Hastie, Tibshirani, Taylor** - *An Introduction to Statistical Learning with Applications in Python* (ISLP), chapter 10: 10.1-10.2 (single and multilayer networks), 10.6 (when to use deep learning), 10.7.1-10.7.4 (fitting: backpropagation, regularisation and stochastic gradient descent, network tuning); 8.2.3 (boosting). Free at statlearning.com.
* **scikit-learn User Guide** - [*Neural network models (supervised)*](https://scikit-learn.org/stable/modules/neural_networks_supervised.html): [the MLP](https://scikit-learn.org/stable/modules/neural_networks_supervised.html#multi-layer-perceptron), [regularisation](https://scikit-learn.org/stable/modules/neural_networks_supervised.html#regularization), [tips on practical use](https://scikit-learn.org/stable/modules/neural_networks_supervised.html#tips-on-practical-use); [*Probability calibration*](https://scikit-learn.org/stable/modules/calibration.html); [*Gradient-boosted trees*](https://scikit-learn.org/stable/modules/ensemble.html#gradient-boosted-trees) (with [monotonic constraints](https://scikit-learn.org/stable/modules/ensemble.html#monotonic-constraints)).
* **Barocas, Hardt, Narayanan** - *Fairness and Machine Learning* (free at [fairmlbook.org](https://fairmlbook.org)): [chapter 3](https://fairmlbook.org/classification.html) ([no fairness through unawareness](https://fairmlbook.org/classification.html#no-fairness-through-unawareness); [the credit-scoring case study](https://fairmlbook.org/classification.html#case-study-credit-scoring); [separation versus sufficiency](https://fairmlbook.org/classification.html#separation-versus-sufficiency)) and [chapter 1](https://fairmlbook.org/introduction.html) ([measurement](https://fairmlbook.org/introduction.html#the-trouble-with-measurement), [feedback loops](https://fairmlbook.org/introduction.html#feedback-and-feedback-loops)).
* **Kleinberg, Mullainathan, Raghavan (2017)** - [*Inherent trade-offs in the fair determination of risk scores*](https://arxiv.org/abs/1609.05807); **Chouldechova (2017)** - [*Fair prediction with disparate impact*](https://arxiv.org/abs/1703.00056).
* **Grinsztajn, Oyallon, Varoquaux (2022)** - [*Why do tree-based models still outperform deep learning on typical tabular data?*](https://arxiv.org/abs/2207.08815)